# Lab 05 — GroupBy & Joins
### Week 2 · Data Engineering for LLM Pipelines

Raw operational data almost never arrives as one tidy table. You get a **fact**
table (orders) and a handful of **dimension** tables (customers, stores), and your
job is to *aggregate* the facts and *join* in the attributes — without silently
corrupting your numbers along the way.

This lab uses a synthetic, clearly-fictional **Cordwell Home & Hardware** B2B
account dataset (generated below — nothing to download).

**By the end you will be able to:**
1. Compute per-key metrics with `groupby().agg(...)` and **named aggregations**
   (`count`, `sum`, `mean`, `nunique`).
2. Choose the correct **join** (inner / left / outer) for a question and **verify
   cardinality** with `validate=`.
3. Diagnose the three classic join failures: **fan-out** (duplicated keys),
   **orphans** (anti-join), and **suffix collisions**.
4. Build tidy, LLM-ready feature rollups and persist them to Parquet.

> **`check()` helper.** Each exercise is followed by soft checks that print ✅/❌.
> Green all the way to `34/34` is your finish line. A red check never stops the
> notebook — it just tells you what's left.


## Setup — build the dataset & the `check()` helper

In [ ]:
%pip install -r requirements.txt

In [ ]:
import warnings
import numpy as np
import pandas as pd

print("pandas", pd.__version__, "| numpy", np.__version__)

def build_cordwell():
    """Synthetic, clearly-fictional Cordwell Home & Hardware B2B account data.

    Returns (customers, orders):
      customers : one row per customer_id  (dimension) — 40 accounts
      orders    : many rows per customer   (fact)      — 200 orders
    The data is deliberately imperfect so the join lessons are real:
      * ~10% of orders reference an unknown customer_id (orphans)
      * 5 customers never place an order (order-less)
      * both tables carry a `region`-style column (a collision waiting to happen)
    """
    rng = np.random.default_rng(2025)
    REGIONS = ["Southeast", "Northeast", "Midwest", "West"]

    n_cust = 40
    cust_ids = [f"C{i:04d}" for i in range(1, n_cust + 1)]
    customers = pd.DataFrame({
        "customer_id": cust_ids,
        "company_name": [f"Cordwell Acct {i:04d}" for i in range(1, n_cust + 1)],
        "region": rng.choice(REGIONS, size=n_cust, p=[.35, .25, .20, .20]),   # home region
    })

    # Only the first 35 customers actually order -> last 5 are order-less.
    ordering_pool = cust_ids[:35]
    n_orders = 200
    real_pick = rng.choice(ordering_pool, size=n_orders)
    orphan_ids = [f"C9{i:03d}" for i in range(1, 21)]           # ids NOT in customers
    is_orphan = rng.random(n_orders) < 0.10
    order_cust = np.where(is_orphan, rng.choice(orphan_ids, size=n_orders), real_pick)

    orders = pd.DataFrame({
        "order_id": np.arange(10001, 10001 + n_orders),
        "customer_id": order_cust,
        "ship_region": rng.choice(REGIONS, size=n_orders),      # where THIS order shipped
        "order_total": np.round(rng.gamma(2.0, 120.0, size=n_orders), 2),
        "freight": np.round(rng.gamma(1.5, 18.0, size=n_orders), 2),
    })
    return customers, orders

customers, orders = build_cordwell()
print("customers:", customers.shape, "| orders:", orders.shape)
display(customers.head(3))
display(orders.head(3))

In [ ]:
# ── soft self-check: prints PASS/FAIL, never raises ──────────────────────────
_score = {"pass": 0, "fail": 0}

def check(label, predicate):
    # `predicate` may be a bool OR a zero-arg callable (lambda). The callable form
    # lets the soft-catch also cover exceptions raised while *computing* the answer,
    # so an unfinished exercise prints FAIL instead of crashing the notebook.
    try:
        ok = bool(predicate() if callable(predicate) else predicate)
        note = ""
    except Exception as e:
        ok, note = False, f"  [error: {type(e).__name__}: {e}]"
    _score["pass" if ok else "fail"] += 1
    print(f"{'✅ PASS' if ok else '❌ FAIL'} — {label}{note}")

def score():
    t = _score["pass"] + _score["fail"]
    print(f"\n{'='*46}\n  {_score['pass']}/{t} checks passing"
          f"  ({_score['fail']} to go)\n{'='*46}")

def raises_merge_error(fn):
    """True iff calling fn() raises pandas' MergeError (used by the fan-out check)."""
    from pandas.errors import MergeError
    try:
        fn()
        return False
    except MergeError:
        return True

---
## Part A — GroupBy fundamentals & named aggregations

`groupby(keys).agg(...)` splits rows into groups, computes one number per group,
and stitches the results back together. **Named aggregations** —
`new_col=("source_col", "func")` — give every output column an explicit name and
source, which is exactly the readable, self-documenting style you want feeding a
downstream feature table.

### A1 — Basic aggregates per `ship_region`

Produce one row per `ship_region` with: order **count**, **mean** freight, and
**total** freight — sorted by order count, descending. Use `as_index=False` so the
group key stays a normal column (not the index).

In [ ]:
# ─── Exercise A1 ───────────────────────────────────────────────
# TODO: one row per ship_region: orders=count(order_id),
#       freight_mean=mean(freight), freight_sum=sum(freight); sort by orders desc.
agg_region = (orders
    .groupby("ship_region", as_index=False)
    .agg(orders=("order_id", "count")))          # placeholder — add the other two aggs
agg_region

In [ ]:
check("A1: one row per ship_region", lambda: len(agg_region) == orders["ship_region"].nunique())
check("A1: has orders/freight_mean/freight_sum columns",
      lambda: {"orders", "freight_mean", "freight_sum"} <= set(agg_region.columns))
check("A1: freight_sum totals to 5190.84",
      lambda: round(float(agg_region["freight_sum"].sum()), 2) == 5190.84)

### A2 — Group by **multiple keys**

Group by `["ship_region", "customer_id"]` and count orders per pair. Grouping by a
list of keys gives you one row per observed **combination**.

In [ ]:
# ─── Exercise A2 ───────────────────────────────────────────────
# TODO: group by BOTH ship_region and customer_id; count orders as n_orders.
agg_pair = orders.groupby("ship_region", as_index=False).agg(n_orders=("order_id", "count"))  # placeholder
print("rows (region×customer combinations):", len(agg_pair))
agg_pair.head()

In [ ]:
check("A2: grouped by two keys", lambda: {"ship_region", "customer_id"} <= set(agg_pair.columns))
check("A2: 121 region×customer combinations", lambda: len(agg_pair) == 121)

### A3 — `size` vs `count` (they are **not** the same)

`size` counts **rows** in each group (nulls included). `count` counts **non-null
values** in a specific column. Watch what happens with an all-missing column:

> **⚠️ CURRENCY FLAG (pandas 3.0).** A column assigned a scalar `None` lands as
> `object` dtype; `count` skips its nulls (→ 0), while `size` still counts the rows.
> Reach for `size` when you want "how many records"; reach for `count` when you want
> "how many have a value here." 

In [ ]:
# ─── Exercise A3 ───────────────────────────────────────────────
# TODO: build `by_size` (group .size()) and `by_count` (.count() of the all-null 'maybe').
tmp = orders.assign(maybe=None)
by_size  = tmp.groupby("ship_region").size()
by_count = tmp.groupby("ship_region").size()          # placeholder — should count 'maybe' instead
print("size  total:", int(by_size.sum()))
print("count total:", int(by_count.sum()))
pd.DataFrame({"size": by_size, "count_maybe": by_count})

In [ ]:
check("A3: size counts all 200 rows", lambda: int(by_size.sum()) == 200)
check("A3: count of all-null column is 0", lambda: int(by_count.sum()) == 0)

---
## Part B — Join patterns & cardinality checks

A join answers "bring me the matching row from the other table." *Which* rows you
keep when there's no match is the whole game:

| join | keeps | typical use |
|---|---|---|
| **inner** | only keys present in **both** | realized facts (orders that have a known customer) |
| **left**  | **all** left rows, NaN where no match | keep every order; flag the unmatched (QA) |
| **outer** | **all** keys from **both** sides | audits — surfaces orphans *and* order-less customers |

### B1 — Inner vs left vs outer, with `validate=`

`orders` is the **many** side (a customer has many orders); `customers` is the
**one** side (one row per `customer_id`). Assert that with
`validate="many_to_one"` — if the "one" side ever has a duplicate key, the merge
raises instead of silently multiplying your rows.

In [ ]:
# ─── Exercise B1 ───────────────────────────────────────────────
# TODO: build inner/left/outer merges of orders+customers on customer_id,
#       each with validate="many_to_one".
inner = orders.merge(customers, on="customer_id", how="inner")   # placeholder — add validate=
left  = inner
outer = inner
print(f"orders={len(orders)}  inner={len(inner)}  left={len(left)}  outer={len(outer)}")
brk = orders.merge(customers, on="customer_id", how="outer", indicator=True)["_merge"].value_counts()
print(brk.to_string())

In [ ]:
check("B1: inner keeps only matched orders (179)", lambda: len(inner) == 179)
check("B1: left keeps every order (200)",          lambda: len(left) == 200)
check("B1: outer surfaces orphans + order-less (205)", lambda: len(outer) == 205)
check("B1: outer = 179 both + 21 left_only + 5 right_only",
      lambda: (len(outer) == len(inner) + (len(left) - len(inner)) + 5) and len(outer) == 205)

### B2 — Anti-join: which orders have **no** customer?

The robust, modern way to find non-matches is `indicator=True` plus a filter on the
`_merge` marker — not fishing for `NaN` in some right-hand column (which breaks the
moment that column legitimately contains nulls).

> These orphaned orders reference a `customer_id` that isn't in the customer master
> — exactly the data-quality signal a `left` join gives you that an `inner` join
> hides.

In [ ]:
# ─── Exercise B2 ───────────────────────────────────────────────
# TODO: left-merge with indicator=True, then keep only the 'left_only' rows.
tagged = orders.merge(customers, on="customer_id", how="left", indicator=True)
orphans = tagged[["order_id", "customer_id"]]            # placeholder — filter to _merge=='left_only'
print("orphan orders:", len(orphans), "| distinct unknown ids:", orphans["customer_id"].nunique())
orphans.head()

In [ ]:
check("B2: found 21 orphan orders", lambda: len(orphans) == 21)
check("B2: from 12 distinct unknown ids", lambda: orphans["customer_id"].nunique() == 12)

### B3 — ⭐ The fan-out trap (the one that silently inflates your numbers)

If the **one** side of a join isn't actually unique, every duplicate key multiplies
the matching left rows — and every downstream `sum`/`count` quietly inflates. It
doesn't crash. It doesn't warn. Your freight total is just *wrong*.

Below, `cust_dupe` has customer **C0001** listed twice (C0001 has 5 orders). Run it
and watch the row count and freight sum climb:

In [ ]:
# a customer master that accidentally lists C0001 twice
cust_dupe = pd.concat([customers, customers.iloc[[0]]], ignore_index=True)

clean  = orders.merge(customers, on="customer_id", how="inner")   # unique right side
fanout = orders.merge(cust_dupe, on="customer_id", how="inner")   # duplicated right side, NO validate

print(f"clean : {len(clean):>3} rows | freight_sum = {clean['freight'].sum():.2f}")
print(f"fanout: {len(fanout):>3} rows | freight_sum = {fanout['freight'].sum():.2f}  <- inflated by C0001's 5 orders")

**Your job:** write a merge that **refuses** to produce that inflated result.
`validate="many_to_one"` makes pandas check the right side is unique *before*
joining and raise `MergeError` if it isn't. Fill in `guarded_merge` so it validates.

In [ ]:
# ─── Exercise B3 ───────────────────────────────────────────────
def guarded_merge(left_df, right_df):
    """Merge that raises MergeError if the right side has duplicate customer_id."""
    # TODO: add validate="many_to_one" so a duplicated right side is rejected.
    return left_df.merge(right_df, on="customer_id", how="inner")

print("clean master ok:", len(guarded_merge(orders, customers)), "rows")

In [ ]:
check("B3: guarded_merge still works on the clean master",
      lambda: len(guarded_merge(orders, customers)) == 179)
check("B3: guarded_merge REJECTS the duplicated master (raises MergeError)",
      lambda: raises_merge_error(lambda: guarded_merge(orders, cust_dupe)))
check("B3: the un-guarded fan-out really did inflate the row count",
      lambda: len(fanout) > len(clean))

### B4 — Suffix collisions

When both sides carry a column of the **same name** that *isn't* a join key, pandas
keeps both and disambiguates with suffixes — defaulting to the unhelpful `_x`/`_y`.
Here `orders` and `customers` both describe a `region` (ship-to vs. home), so name
them explicitly with `suffixes=`.

In [ ]:
# ─── Exercise B4 ───────────────────────────────────────────────
orders_v2 = orders.rename(columns={"ship_region": "region"})
# TODO: merge orders_v2 + customers; give the colliding region columns
#       readable names via suffixes=("_ship", "_home").
auto = orders_v2.merge(customers, on="customer_id", how="inner")
fixed = orders_v2.merge(customers, on="customer_id", how="inner")           # placeholder — add suffixes=
print("default suffixes:", [c for c in auto.columns  if c.startswith("region")])
print("named suffixes  :", [c for c in fixed.columns if c.startswith("region")])
fixed.filter(items=["customer_id", "region_ship", "region_home"]).head()

In [ ]:
check("B4: default merge produced region_x / region_y",
      lambda: {"region_x", "region_y"} <= set(auto.columns))
check("B4: named suffixes produced region_ship / region_home",
      lambda: {"region_ship", "region_home"} <= set(fixed.columns))

---
## Part C — End-to-end: customer segments & region rollups

Now assemble the moves into a real deliverable: per-customer metrics, joined to
customer attributes, bucketed into spend segments, and rolled up by region — the
kind of tidy feature table you'd hand to a downstream prompt or model.

### C1 — Per-customer aggregates

One row per `customer_id`: order count, total spend, total freight.

In [ ]:
# ─── Exercise C1 ───────────────────────────────────────────────
# TODO: per customer_id -> n_orders (count), total_spend (sum order_total),
#       freight_sum (sum freight).
per_cust = (orders
    .groupby("customer_id", as_index=False)
    .agg(n_orders=("order_id", "count")))            # placeholder — add total_spend, freight_sum
print("per-customer rows (includes unknown/orphan ids):", len(per_cust))
per_cust.head()

In [ ]:
check("C1: has n_orders/total_spend/freight_sum",
      lambda: {"n_orders", "total_spend", "freight_sum"} <= set(per_cust.columns))
check("C1: 47 rows (35 real customers + 12 unknown ids)", lambda: len(per_cust) == 47)

### C2 — Join attributes, then segment

Join `per_cust` to `customers`. **inner** keeps only customers we have attributes
for (drops the unknown-id aggregates); **left** keeps every aggregate but leaves
attributes `NaN` for the unknowns. For a segment *report* we need attributes, so we
build on the **inner** result — then bucket `total_spend` into low/mid/high with
`pd.cut`.

Because both frames are one-row-per-customer, assert `validate="one_to_one"`.

In [ ]:
# ─── Exercise C2 ───────────────────────────────────────────────
# TODO: inner- AND left-join per_cust to customers (validate="one_to_one"),
#       then add spend_segment = pd.cut(total_spend, [0,600,1100,inf], low/mid/high, right=False).
per_cust_inner = per_cust.merge(customers, on="customer_id", how="inner")   # placeholder — add validate
per_cust_left  = per_cust_inner
print(f"inner={len(per_cust_inner)}  |  left={len(per_cust_left)}")
per_cust_inner = per_cust_inner.assign(spend_segment=pd.NA)                  # placeholder — use pd.cut
per_cust_inner.head()

In [ ]:
check("C2: inner drops unknown ids -> 35 rows", lambda: len(per_cust_inner) == 35)
check("C2: left keeps all 47, 12 with NaN attributes",
      lambda: len(per_cust_left) == 47 and int(per_cust_left["company_name"].isna().sum()) == 12)
check("C2: spend_segment is an ordered low/mid/high category",
      lambda: list(per_cust_inner["spend_segment"].cat.categories) == ["low", "mid", "high"])
check("C2: segment counts are 7 / 10 / 18",
      lambda: per_cust_inner.groupby("spend_segment", observed=True).size().tolist() == [7, 10, 18])

### C2b — ⚠️ CURRENCY FLAG: `observed=` when grouping by a **categorical**

`spend_segment` is a categorical. In **pandas 3.0 the default is `observed=True`** —
empty categories are dropped from the result. Older tutorials assume `observed=False`
(every category shown, even with count 0). Slice to one small region to see the
difference, and pass `observed=` explicitly so your intent is unambiguous.

In [ ]:
midwest = per_cust_inner[per_cust_inner["region"] == "Midwest"]
print("Midwest customers:", len(midwest), "->", list(midwest["spend_segment"]))
print("\nobserved=True  (pandas 3.0 default — empty 'low' dropped):")
print(midwest.groupby("spend_segment", observed=True).size().to_string())
print("\nobserved=False (every category kept, even count 0):")
print(midwest.groupby("spend_segment", observed=False).size().to_string())

In [ ]:
check("C2b: observed=True yields fewer groups than observed=False here",
      lambda: midwest.groupby("spend_segment", observed=True).size().shape[0]
            < midwest.groupby("spend_segment", observed=False).size().shape[0])

### C3 — Region rollup for reporting

Roll the per-customer table up to one row per **home region**: how many customers,
how many orders, and total spend — sorted by orders.

In [ ]:
# ─── Exercise C3 ───────────────────────────────────────────────
# TODO: per home region -> customers=count(customer_id), orders=sum(n_orders),
#       total_spend=sum(total_spend); sort by orders desc.
region_rollup = per_cust_inner[["region"]].drop_duplicates(ignore_index=True)  # placeholder — add the aggs
region_rollup

In [ ]:
check("C3: one row per region (4)", lambda: len(region_rollup) == 4)
check("C3: customers sum back to 35", lambda: int(region_rollup["customers"].sum()) == 35)
check("C3: orders sum back to 179",   lambda: int(region_rollup["orders"].sum()) == 179)

---
## Part D — Bonus: scale up with **partitioned Parquet** (self-contained)

The same `groupby`/rollup patterns scale to millions of rows, and Parquet lets you
**partition** data on disk by a column so downstream readers only touch the slices
they need. Generate a larger synthetic orders table, write it partitioned by
`ship_region`, read it back, and repeat the rollup — all in-notebook, no external
files required.

> **⚠️ CURRENCY FLAG.** Reading back a partitioned dataset returns the partition
> column (`ship_region`) as a **categorical** (its values came from directory
> names). That's expected — cast to `str` if a downstream step needs a plain string
> column.

In [ ]:
# ─── Exercise D ────────────────────────────────────────────────
from pathlib import Path
import shutil

rng_big = np.random.default_rng(7)
N = 5_000
big = pd.DataFrame({
    "order_id": np.arange(1, N + 1),
    "customer_id": rng_big.choice(customers["customer_id"], size=N),
    "ship_region": rng_big.choice(["Southeast", "Northeast", "Midwest", "West"], size=N),
    "order_total": np.round(rng_big.gamma(2.0, 120.0, size=N), 2),
})

out = Path("artifacts/orders_big")
out.parent.mkdir(parents=True, exist_ok=True)
if out.exists():
    shutil.rmtree(out) if out.is_dir() else out.unlink()
# TODO: write `big` to `out` partitioned by ship_region, then read it back into `back`
#       and build big_rollup (orders=count, total_spend=sum) per ship_region.
big.to_parquet(out)                                          # placeholder — add partition_cols=["ship_region"]
parts = sorted(p.name for p in out.glob("ship_region=*"))
back = pd.read_parquet(out)
big_rollup = back.groupby("ship_region", as_index=False, observed=True).agg(orders=("order_id", "count"))
print("partitions on disk:", parts)
print("rows read back:", len(back))
big_rollup

In [ ]:
check("D: wrote 4 ship_region partitions", lambda: len(parts) == 4)
check("D: read back all 5,000 rows", lambda: len(back) == 5000)
check("D: rollup orders sum back to 5,000", lambda: int(big_rollup["orders"].sum()) == 5000)

---
## Part E — Wrap-up & export

Persist the two deliverables so a downstream lab can pick them up. Then answer the
reflection questions in the final Markdown cell.

In [ ]:
# ─── Exercise E ────────────────────────────────────────────────
from pathlib import Path
clean_dir = Path("artifacts/clean")
clean_dir.mkdir(parents=True, exist_ok=True)

# TODO: write per_cust_inner AND region_rollup to parquet here (index=False) ...
# TODO: ... then read per_customer.parquet back into `rt`.
rt = pd.DataFrame()                                         # placeholder — replace with the round-trip read
print("wrote per_customer.parquet:", rt.shape)

In [ ]:
check("E: per_customer.parquet round-trips to 35 rows", lambda: rt.shape[0] == 35)
check("E: per_customer.parquet kept the spend_segment column", lambda: "spend_segment" in rt.columns)
check("E: region_rollup.parquet has 4 rows",
      lambda: pd.read_parquet(Path("artifacts/clean") / "region_rollup.parquet").shape[0] == 4)
score()

---
## Wrap-up — answer in this Markdown cell

1. Give one question where an **inner** join is the right choice, and one where a
   **left** join is required. *(Hint: realized revenue vs. data-quality audit.)*
2. Show — in words — how `merge(..., validate="many_to_one")` protects a freight
   total from a fan-out **before** the numbers are ever computed.
3. When would you group with `observed=False` instead of the pandas-3.0 default?

**Key takeaways**
- Named aggregations make feature tables self-documenting: `col=("src", "func")`.
- Pick the join by what you do with non-matches: **inner** = realized facts,
  **left** = keep-everything + QA, **outer** = audit both sides.
- `validate=` is a cheap guardrail against the silent fan-out that inflates metrics.
- `size` counts rows; `count` counts non-null values — they diverge on missing data.
- Grouping by a categorical drops empty groups by default in pandas 3.0
  (`observed=True`); pass it explicitly when it matters.
